In [ ]:
!pip install -U transformers sentence-transformers tqdm

In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from dateutil import parser

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer  

In [ ]:

# =====================
# CONFIG
# =====================
RAW_FOLDER = "/home/sunkari/Stock_price_predictor/Dataset"
OUTPUT_FOLDER = "./processed_datasets"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Use GPU only if available and mostly free. Otherwise CPU.
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
print("Device:", DEVICE)

# =====================
# Load models safely
# =====================
# Lightweight sentence-transformer for embeddings (fast + memory friendly)
EMB_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"  # 768-dim
# Finance sentiment classifier
SENT_MODEL_NAME = "yiyanghkust/finbert-tone"

# Load embedding model (SentenceTransformer handles device internally)
print("Loading embedding model:", EMB_MODEL_NAME)
emb_model = SentenceTransformer(EMB_MODEL_NAME, device=str(DEVICE))  # will be on CPU or GPU
emb_model.max_seq_length = 128

print("Loading sentiment model:", SENT_MODEL_NAME)
sent_tokenizer = AutoTokenizer.from_pretrained(SENT_MODEL_NAME)
sent_model = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL_NAME)
# Put sentiment model on CPU by default, or GPU if you have enough free memory:
try:
    if USE_CUDA:
        # cautiously try GPU
        sent_model.to(DEVICE)
        print("Sentiment model moved to GPU.")
    else:
        sent_model.to("cpu")
except RuntimeError:
    # fallback to CPU
    torch.cuda.empty_cache()
    sent_model.to("cpu")
    DEVICE = torch.device("cpu")
    print("Fell back to CPU for sentiment model.")

sent_model.eval()

# =====================
# Utilities
# =====================
def normalize_date_column(df):
    def parse_date_safe(x):
        try:
            return parser.parse(str(x), dayfirst=False)
        except:
            try:
                return parser.parse(str(x), dayfirst=True)
            except:
                return pd.NaT
    df["Date"] = df["Date"].apply(parse_date_safe)
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
    return df

def split_headlines(text):
    if pd.isna(text):
        return []
    return [t.strip() for t in str(text).split('|') if t.strip()]

# batch inference helpers
@torch.inference_mode()
def compute_sentiment_scores(texts, batch_size=16, device=DEVICE):
    """Return array of sentiment scores (positive_prob - negative_prob) for each text."""
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = sent_tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
        # move tensors to device where model is
        model_device = next(sent_model.parameters()).device
        inputs = {k: v.to(model_device) for k, v in inputs.items()}
        out = sent_model(**inputs)
        probs = torch.nn.functional.softmax(out.logits, dim=-1).cpu().numpy()  # shape (B, 3)
        # finbert-tone ordering: check model card, but common: [neg, neu, pos]
        # score = pos - neg
        batch_scores = probs[:, 2] - probs[:, 0]
        scores.extend(batch_scores.tolist())
        # free GPU
        if model_device.type == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
    return np.array(scores, dtype=np.float32)

def compute_embeddings(texts, batch_size=32):
    """
    Use sentence-transformers model which handles batching internally.
    Returns array shape (len(texts), emb_dim)
    """
    # SentenceTransformer.encode supports batching and returns numpy array
    embs = emb_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        max_length=128,
        truncation=True
    )
    return embs  # (N, D)

# =====================
# Main loop
# =====================
files = [f for f in os.listdir(RAW_FOLDER) if f.endswith(".csv")]
for fname in tqdm(files, desc="Processing CSV files"):
    path = os.path.join(RAW_FOLDER, fname)
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
    df = normalize_date_column(df)
    if "Headlines" not in df.columns:
        print(f"Skipping {fname}: no Headlines column")
        continue

    df["Headline_List"] = df["Headlines"].apply(split_headlines)

    all_embs = []
    all_sentiments = []

    # We'll collect all headline texts for batch encoding / sentiment, but per-row groups vary.
    # Approach: flatten (row_idx, headline) pairs -> compute embeddings and sentiment for all headlines -> group back.
    row_headlines = []   # flat list of headline strings
    row_index_map = []   # maps each headline to row idx

    for idx, hl_list in enumerate(df["Headline_List"].tolist()):
        if not hl_list:
            continue
        for h in hl_list:
            row_headlines.append(h)
            row_index_map.append(idx)

    if len(row_headlines) == 0:
        # no headlines at all; fill zeros
        emb_dim = emb_model.get_sentence_embedding_dimension()
        zeros = np.zeros((len(df), emb_dim), dtype=np.float32)
        df_out = pd.concat([df.drop(columns=["Headline_List"]), pd.DataFrame(zeros, columns=[f"emb_{i}" for i in range(zeros.shape[1])])], axis=1)
        df_out["sentiment_score"] = 0.0
        out_path = os.path.join(OUTPUT_FOLDER, fname.replace(".csv", "_merged.csv"))
        df_out.to_csv(out_path, index=False)
        continue

    # Compute embeddings for all headlines in batches (memory friendly)
    headline_embeddings = compute_embeddings(row_headlines, batch_size=32)  # shape (total_headlines, emb_dim)

    # Compute sentiment scores for all headlines (uses sentiment model)
    headline_sentiments = compute_sentiment_scores(row_headlines, batch_size=32)  # shape (total_headlines,)

    # Now aggregate per-row: mean embedding and mean sentiment across headlines for that row
    emb_dim = headline_embeddings.shape[1]
    per_row_embs = np.zeros((len(df), emb_dim), dtype=np.float32)
    per_row_sents = np.zeros(len(df), dtype=np.float32)
    counts = np.zeros(len(df), dtype=np.int32)

    for i, row_idx in enumerate(row_index_map):
        per_row_embs[row_idx] += headline_embeddings[i]
        per_row_sents[row_idx] += headline_sentiments[i]
        counts[row_idx] += 1

    nonzero = counts > 0
    per_row_embs[nonzero] = per_row_embs[nonzero] / counts[nonzero][:, None]
    per_row_sents[nonzero] = per_row_sents[nonzero] / counts[nonzero]

    # For rows without headlines, keep zeros and sentiment 0.0 (neutral)
    emb_cols = [f"emb_{i}" for i in range(emb_dim)]
    emb_df = pd.DataFrame(per_row_embs, columns=emb_cols)
    df["sentiment_score"] = per_row_sents

    df_out = pd.concat([df.drop(columns=["Headline_List"]), emb_df], axis=1)
    out_path = os.path.join(OUTPUT_FOLDER, fname.replace(".csv", "_merged.csv"))
    df_out.to_csv(out_path, index=False)

    # cleanup per-file
    del headline_embeddings, headline_sentiments, row_headlines, row_index_map
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Done.")

In [ ]:
import os, glob, pickle, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

# ===========================
# CONFIG
# ===========================
WINDOW_SIZE = 8
INPUT_FOLDER = "./processed_datasets"
NORMALIZED_FOLDER = "./normalized_datasets"
SAVE_FOLDER = "./windows"
SCALER_FOLDER = "./scalers"

os.makedirs(NORMALIZED_FOLDER, exist_ok=True)
os.makedirs(SAVE_FOLDER, exist_ok=True)
os.makedirs(SCALER_FOLDER, exist_ok=True)

random.seed(42)
np.random.seed(42)

# ===========================
# PROCESSING PIPELINE
# ===========================
files = glob.glob(os.path.join(INPUT_FOLDER, "*_merged.csv"))
all_windows, companies = [], []

for f in tqdm(files):
    df = pd.read_csv(f, parse_dates=["Date"])
    ticker = df["Ticker"].iloc[0] if "Ticker" in df.columns else os.path.basename(f).split("_")[0]

    # --- Drop unwanted columns ---
    drop_cols = [c for c in df.columns if c.startswith("emb_") or "headline" in c.lower()]
    df = df.drop(columns=drop_cols, errors="ignore")

    # --- Select numeric columns ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 0:
        print(f"⚠️ Skipping {ticker}: no numeric columns found.")
        continue
    if "Close" not in df.columns:
        print(f"⚠️ Skipping {ticker}: 'Close' column missing.")
        continue

    # --- Normalize numeric columns per company ---
    scaler = StandardScaler()
    exclude_cols = ["sentiment_score"]  # keep sentiment as-is
    numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

    # --- Save normalized dataset ---
    norm_path = os.path.join(NORMALIZED_FOLDER, f"{ticker}_normalized.csv")
    df.to_csv(norm_path, index=False)

    # --- Save scaler for later denormalization ---
    with open(os.path.join(SCALER_FOLDER, f"{ticker}_scaler.pkl"), "wb") as handle:
        pickle.dump(scaler, handle)

    # --- Create time-series windows ---
    numeric_df = df[numeric_cols]
    for i in range(len(df) - WINDOW_SIZE):
        X = numeric_df.iloc[i:i+WINDOW_SIZE].values.astype(np.float32)
        y = float(df["Close"].iloc[i+WINDOW_SIZE])  # normalized Close
        all_windows.append((X, y, ticker))

    companies.append(ticker)

# ===========================
# COMBINE + SPLIT
# ===========================
random.shuffle(all_windows)
split = int(0.8 * len(all_windows))
train_windows = all_windows[:split]
test_windows = all_windows[split:]

# ===========================
# SAVE
# ===========================
pickle.dump(train_windows, open(f"{SAVE_FOLDER}/train_windows.pkl", "wb"))
pickle.dump(test_windows, open(f"{SAVE_FOLDER}/test_windows.pkl", "wb"))
pickle.dump(sorted(list(set(companies))), open(f"{SAVE_FOLDER}/company_list.pkl", "wb"))

print(f"✅ Processed {len(companies)} companies")
print(f"✅ Created {len(train_windows)} train and {len(test_windows)} test windows")
print(f"✅ All data and scalers saved successfully!")

In [4]:
import os
import glob
import pickle
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

# ===========================
# CONFIGURATION
# ===========================
WINDOW_SIZE = 8
NORMALIZED_FOLDER = "./normalized_datasets"
SAVE_FOLDER = "./windows"

os.makedirs(SAVE_FOLDER, exist_ok=True)

random.seed(42)
np.random.seed(42)

# ===========================
# BUILD WINDOWS
# ===========================
files = glob.glob(os.path.join(NORMALIZED_FOLDER, "*_normalized.csv"))
all_windows, companies = [], []

for f in tqdm(files, desc="Building time-series windows"):
    df = pd.read_csv(f, parse_dates=["Date"])
    ticker = df["Ticker"].iloc[0] if "Ticker" in df.columns else os.path.basename(f).split("_")[0]

    # --- Ensure 'Close' column exists ---
    if "Close" not in df.columns:
        print(f"⚠️ Skipping {ticker}: 'Close' column missing.")
        continue

    # --- Keep only numeric columns ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 0:
        print(f"⚠️ Skipping {ticker}: no numeric columns found.")
        continue

    # --- Create time-series windows ---
    numeric_df = df[numeric_cols]
    for i in range(len(df) - WINDOW_SIZE):
        X = numeric_df.iloc[i:i + WINDOW_SIZE].values.astype(np.float32)
        y = float(df["Close"].iloc[i + WINDOW_SIZE])  # normalized Close
        all_windows.append((X, y, ticker))

    companies.append(ticker)

# ===========================
# COMBINE + SPLIT
# ===========================
random.shuffle(all_windows)
split_idx = int(0.8 * len(all_windows))
train_windows = all_windows[:split_idx]
test_windows = all_windows[split_idx:]

# ===========================
# SAVE DATASETS
# ===========================
pickle.dump(train_windows, open(os.path.join(SAVE_FOLDER, "train_windows.pkl"), "wb"))
pickle.dump(test_windows, open(os.path.join(SAVE_FOLDER, "test_windows.pkl"), "wb"))
pickle.dump(sorted(list(set(companies))), open(os.path.join(SAVE_FOLDER, "company_list.pkl"), "wb"))

print(f"✅ Processed {len(companies)} companies")
print(f"✅ Created {len(train_windows)} train and {len(test_windows)} test windows")
print(f"✅ All windows saved successfully to '{SAVE_FOLDER}'")


Building time-series windows: 100%|██████████| 10/10 [00:00<00:00, 11.54it/s]


✅ Processed 10 companies
✅ Created 7976 train and 1994 test windows
✅ All windows saved successfully to './windows'


In [4]:
import pandas as pd
import numpy as np
import pickle
import random
import os
import glob
from tqdm import tqdm

# --- Configuration ---
INPUT_DIR = "normalized_datasets"  # ⚠️ Directory with all your CSVs
OUTPUT_DIR = "windows"
WINDOW_SIZE = 8      # 8 days of features to predict the 9th day
TEST_SPLIT_RATIO = 0.1 # 10% of data for testing, 90% for training
TARGET_COLUMN = 'Adj Close'
# ---------------------

# These are all the columns your model will use as features
feature_cols = [
    'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 
    'Daily_Return', 'EMA_7', 'EMA_21', 
    'negative', 'neutral', 'positive'
]

# --- 1. Load and Concatenate All CSVs ---
csv_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
if not csv_files:
    print(f"❌ ERROR: No CSV files found in '{INPUT_DIR}' directory.")
    print("Please check the directory name and that your CSVs are inside it.")
    exit()

print(f"Found {len(csv_files)} CSV files. Loading and combining...")

all_dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        all_dfs.append(df)
    except Exception as e:
        print(f"Warning: Could not read {f}. Error: {e}")

if not all_dfs:
    print("❌ ERROR: No data was successfully loaded.")
    exit()

master_df = pd.concat(all_dfs, ignore_index=True)
print(f"✅ Combined all files into one DataFrame with {len(master_df)} rows.")

# --- 2. Pre-process Combined Data ---
# Handle missing sentiment data (the ',,,' in your sample)
master_df[['negative', 'neutral', 'positive']] = master_df[['negative', 'neutral', 'positive']].fillna(0)

# Make sure date is in the correct format for sorting
master_df['Date'] = pd.to_datetime(master_df['Date'])

all_windows = []
all_tickers = master_df['Ticker'].unique()

print(f"Found {len(all_tickers)} companies. Creating {WINDOW_SIZE}-day windows...")

# --- 3. Create Windows (Grouped by Ticker) ---
for ticker in tqdm(all_tickers, desc="Processing tickers"):
    
    # Get all data for one company and sort by date
    company_df = master_df[master_df['Ticker'] == ticker].sort_values(by='Date')
    
    # Skip if company doesn't have enough data
    if len(company_df) <= WINDOW_SIZE:
        continue
        
    # Convert data to numpy for speed
    feature_data = company_df[feature_cols].values
    target_data = company_df[TARGET_COLUMN].values
    
    # Create windows
    # We need (WINDOW_SIZE) days for features + 1 day for target
    # So we iterate up to len(company_df) - WINDOW_SIZE
    for i in range(len(company_df) - WINDOW_SIZE):
        
        # Features: e.g., Day 1 to Day 8
        window_X = feature_data[i : i + WINDOW_SIZE]
        
        # Target: e.g., Day 9
        target_y = target_data[i + WINDOW_SIZE]
        
        # ⚠️ THIS IS THE CRITICAL FIX ⚠️
        # This solves our 'NaN' and '0.0' loss problems from earlier.
        # We ensure no 'NaN' data gets into our samples.
        if np.isnan(window_X).any() or np.isnan(target_y):
            continue
            
        all_windows.append((window_X, target_y, ticker))

print(f"\nCreated {len(all_windows)} total valid windows.")

# --- 4. Mix, Shuffle, and Split ---
print("Shuffling data...")
random.shuffle(all_windows)

print("Splitting into train and test sets...")
split_index = int(len(all_windows) * (1 - TEST_SPLIT_RATIO))

train_windows = all_windows[:split_index]
test_windows = all_windows[split_index:]

print(f"  > Training samples: {len(train_windows)}")
print(f"  > Testing samples:  {len(test_windows)}")

# --- 5. Save Files ---
print(f"Saving files to '{OUTPUT_DIR}' directory...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the 3 files your training script will need
with open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "wb") as f:
    pickle.dump(train_windows, f)
    
with open(os.path.join(OUTPUT_DIR, "test_windows.pkl"), "wb") as f:
    pickle.dump(test_windows, f)
    
with open(os.path.join(OUTPUT_DIR, "company_list.pkl"), "wb") as f:
    pickle.dump(list(all_tickers), f)

print("\n" + "="*40)
print("✅ Preprocessing complete!")
print(f"Files saved in '{OUTPUT_DIR}':")
print("  - train_windows.pkl")
print("  - test_windows.pkl")
print("  - company_list.pkl")
print("="*40)

Found 10 CSV files. Loading and combining...
✅ Combined all files into one DataFrame with 10050 rows.
Found 10 companies. Creating 8-day windows...


Processing tickers: 100%|██████████| 10/10 [00:00<00:00, 116.81it/s]


Created 9713 total valid windows.
Shuffling data...
Splitting into train and test sets...
  > Training samples: 8741
  > Testing samples:  972
Saving files to 'windows' directory...



✅ Preprocessing complete!
Files saved in 'windows':
  - train_windows.pkl
  - test_windows.pkl
  - company_list.pkl


## final generation

In [ ]:
import os

import time

import logging

import torch

import torch.nn.functional as F

import pandas as pd

from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from dateutil import parser



# =====================================================

LOG_FILE = "processing_log.txt"

logging.basicConfig(

    filename=LOG_FILE,

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    datefmt="%Y-%m-%d %H:%M:%S"

)



def log(msg, level="info"):

    tqdm.write(msg)

    if level == "error":

        logging.error(msg)

    elif level == "warning":

        logging.warning(msg)

    else:

        logging.info(msg)





# =====================================================

def normalize_date_column(df):

    def parse_date_safe(x):

        try:

            return parser.parse(str(x), dayfirst=False)

        except Exception:

            try:

                return parser.parse(str(x), dayfirst=True)

            except Exception:

                return None



    df["Date"] = df["Date"].apply(parse_date_safe)

    df = df.dropna(subset=["Date"])

    df = df.sort_values("Date").reset_index(drop=True)

    return df





def split_headlines(text):

    if pd.isna(text):

        return []

    return [t.strip() for t in str(text).split('|') if t.strip()]





# =====================================================

# --- Use your raw dataset path here ---

DATA_DIR = "/home/sunkari/Stock_price_predictor/Dataset" 

OUTPUT_DIR = "./Processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)



TRAIN_RATIO = 0.8  # 80% train, 20% test



# =====================================================

MODEL_NAME = "yiyanghkust/finbert-tone"

log(f"🔹 Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.eval()

log("✅ Model loaded successfully.")





# =====================================================

def get_sentiment_scores(texts):

    if len(texts) == 0:

        return [0.0, 1.0, 0.0]  # Neutral default

    inputs = tokenizer(

        texts,

        padding=True,

        truncation=True,

        max_length=128,

        return_tensors="pt"

    )

    with torch.no_grad():

        outputs = model(**inputs)

        probs = F.softmax(outputs.logits, dim=-1)

    return probs.mean(dim=0).numpy().tolist()





# =====================================================

start_time_total = time.time()



for file in os.listdir(DATA_DIR):

    if not file.endswith(".csv"):

        continue

    start_time = time.time()

    company_path = os.path.join(DATA_DIR, file)

    log(f"\n🔍 Processing {file} ...")



    try:

        df = pd.read_csv(company_path)

        df.columns = df.columns.str.strip()

        log(f"📂 Loaded file with {len(df)} rows.")



        df = normalize_date_column(df)

        log(f"🗓️ Normalized dates, {len(df)} rows remain after cleaning.")



        # Split headlines into lists

        df["Headline_List"] = df["Headlines"].apply(split_headlines)



        # Compute daily sentiment

        sentiments = []

        for headlines in tqdm(df["Headline_List"].tolist(), desc=f"Sentiment {file}"):

            try:

                probs = get_sentiment_scores(headlines)

                sentiments.append(probs)

            except Exception as e:

                log(f"⚠️ Error processing headlines: {headlines[:3]}... | {e}", "warning")

                sentiments.append([0.0, 1.0, 0.0])  # default neutral



        sentiments = pd.DataFrame(sentiments, columns=["negative", "neutral", "positive"])

        df = pd.concat([df, sentiments], axis=1)



        # --- 🧹 CLEANUP (As Requested) ---

        # Drop the text columns after processing
        df = df.drop(columns=["Headlines", "Headline_List"], errors='ignore')
        
        # Save processed versions

        base_name = os.path.splitext(file)[0]

        train_out = os.path.join(OUTPUT_DIR, f"{base_name}_train.csv")



        df.to_csv(train_out, index=False)

        elapsed = time.time() - start_time

        log(f"✅ Saved data split for {file} | data={len(df)} | ⏱️ {elapsed:.2f}s")



    except Exception as e:

        log(f"❌ Error processing {file}: {e}", "error")



total_time = time.time() - start_time_total

log(f"\n🏁 All files processed successfully in {total_time/60:.2f} minutes.")

🔹 Loading model: yiyanghkust/finbert-tone
✅ Model loaded successfully.

🔍 Processing XOM_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment XOM_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:03<00:00, 15.78it/s]


✅ Saved data split for XOM_stock_gdelt_final.csv | data=1003 | ⏱️ 63.66s

🔍 Processing MSFT_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment MSFT_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:36<00:00, 10.44it/s]


✅ Saved data split for MSFT_stock_gdelt_final.csv | data=1003 | ⏱️ 96.16s

🔍 Processing V_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


...

✅ Saved data split for TSLA_stock_gdelt_final.csv | data=1003 | ⏱️ 105.81s

🔍 Processing JPM_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment JPM_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:42<00:00,  9.81it/s]


✅ Saved data split for JPM_stock_gdelt_final.csv | data=1003 | ⏱️ 102.40s

🔍 Processing AAPL_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment AAPL_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:23<00:00, 12.06it/s]

✅ Saved data split for AAPL_stock_gdelt_final.csv | data=1003 | ⏱️ 83.26s

🏁 All files processed successfully in 14.80 minutes.


In [5]:
import pandas as pd
import numpy as np
import pickle
import random
import os
import glob
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

# --- Configuration ---
INPUT_DIR = "Processed"
OUTPUT_DIR = "windows"
SCALER_DIR = "scalers"
WINDOW_SIZE = 8
TARGET_COLUMN = 'Close'
TEST_SPLIT_RATIO = 0.1 # 10% for test, 90% for train
# ---------------------

# --- 1. SEPARATE FEATURE LISTS ---
financial_features = [
    'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 
    'Daily_Return', 'EMA_7', 'EMA_21'
]
sentiment_features = ['negative', 'neutral', 'positive']

TARGET_COL_INDEX = 0 # 'Adj Close' is 0th in financial_features
NUM_FINANCIAL_FEATURES = len(financial_features)

print(f"Normalizing {NUM_FINANCIAL_FEATURES} features: {financial_features}")
print(f"Skipping {len(sentiment_features)} features: {sentiment_features}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALER_DIR, exist_ok=True)

# --- 2. Get All Processed CSVs ---
all_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
if not all_files:
    print(f"❌ ERROR: No '*.csv' files found in '{INPUT_DIR}'.")
    print("Please run '01_process_headlines.py' first.")
    exit()

print(f"Found {len(all_files)} company files to process.")

all_windows = [] 
all_tickers = []

# --- 3. Process Each Company ---
for file in tqdm(all_files, desc="Processing Companies"):
    ticker = os.path.basename(file).split('.')[0]
    all_tickers.append(ticker)
    
    df = pd.read_csv(file)
    
    # --- 3a. Fit and Save Scaler ---
    df_clean = df.dropna(subset=financial_features)
    
    scaler = StandardScaler()
    scaler.fit(df_clean[financial_features])
    
    scaler_path = os.path.join(SCALER_DIR, f"{ticker}_scaler.pkl")
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)

    # --- 3b. Create Windows ---
    df[sentiment_features] = df[sentiment_features].fillna(0.0)
    
    if len(df_clean) <= WINDOW_SIZE:
        continue
        
    # Normalize financial features
    features_financial_norm = scaler.transform(df_clean[financial_features])
    features_sentiment_orig = df_clean[sentiment_features].values
    
    features_combined = np.concatenate([features_financial_norm, features_sentiment_orig], axis=1)
    
    for i in range(len(features_combined) - WINDOW_SIZE):
        window_X = features_combined[i : i + WINDOW_SIZE]
        target_y = features_combined[i + WINDOW_SIZE, TARGET_COL_INDEX]
        
        if not (np.isnan(window_X).any() or np.isnan(target_y)):
            all_windows.append((window_X, target_y, ticker))

# --- 4. Mix, Shuffle, and Split (as requested) ---
print(f"\nCreated {len(all_windows)} total windows from {len(all_tickers)} companies.")
print(f"✅ Saved {len(all_tickers)} scalers to '{SCALER_DIR}'.")

print("Shuffling all windows...")
random.shuffle(all_windows)

print("Splitting into train and test sets...")
split_index = int(len(all_windows) * (1 - TEST_SPLIT_RATIO))

train_windows = all_windows[:split_index]
test_windows = all_windows[split_index:]

print(f"  > Training samples: {len(train_windows)}")
print(f"  > Testing samples:  {len(test_windows)}")

# --- 5. Save Files ---
print(f"Saving files to '{OUTPUT_DIR}' directory...")
with open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "wb") as f:
    pickle.dump(train_windows, f)
    
with open(os.path.join(OUTPUT_DIR, "test_windows.pkl"), "wb") as f:
    pickle.dump(test_windows, f)
    
with open(os.path.join(OUTPUT_DIR, "company_list.pkl"), "wb") as f:
    pickle.dump(all_tickers, f)

print("\n" + "="*40)
print("✅ Preprocessing complete!")
print("="*40)

Normalizing 9 features: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return', 'EMA_7', 'EMA_21']
Skipping 3 features: ['negative', 'neutral', 'positive']
Found 10 company files to process.


Processing Companies: 100%|██████████| 10/10 [00:00<00:00, 59.42it/s]


Created 9910 total windows from 10 companies.
✅ Saved 10 scalers to 'scalers'.
Shuffling all windows...
Splitting into train and test sets...
  > Training samples: 8919
  > Testing samples:  991
Saving files to 'windows' directory...



✅ Preprocessing complete!
